# UC-DIM-3 — Pentadal Rainfall Dimension Pagination

**Persona:** GIS Specialist — CHIRPS Rainfall Analyst

**Goal:** Page through all 72 pentadal periods of year 2024 for a CHIRPS rainfall
collection, following `links[rel=next]` tokens until exhausted. Verify the February
edge case (period P6 has variable day count). Confirm that an oversize `limit` is
clamped by the server rather than rejected with 422.

**Use case:** UC-DIM-3 — CHIRPS pentadal precipitation monitoring

**Prerequisites:**
- `extension_dimensions` installed and registered
- Target collection (`chirps_pentadal`) has a temporal dimension with `pentadal` encoding
- Env vars: `DYNASTORE_BASE_URL`, `DYNASTORE_TOKEN`, `CATALOG_ID`
- Optional: `COLLECTION_ID` (defaults to `chirps_pentadal`)

In [ ]:
import os

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ["DYNASTORE_BASE_URL"]
TOKEN = os.environ["DYNASTORE_TOKEN"]
CATALOG_ID = os.environ["CATALOG_ID"]
COLLECTION_ID = os.environ.get("COLLECTION_ID", "chirps_pentadal")

DIM_BASE = f"/dimensions/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/dimensions"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0)
print(f"BASE_URL     : {BASE_URL}")
print(f"CATALOG_ID   : {CATALOG_ID}")
print(f"COLLECTION_ID: {COLLECTION_ID}")

## Step 1 — Introspect dimension encoding

Confirm the `time` dimension declares `pentadal` encoding before attempting to
retrieve 72 periods. Pentadal calendars divide each month into 6 pentads of
approximately 5 days each (P1–P6), giving 72 periods per year.

In [ ]:
resp = client.get(f"{DIM_BASE}/time")
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

time_dim = resp.json()
encoding = time_dim.get("encoding")
dim_type = time_dim.get("type", "")

print(f"id      : {time_dim.get('id')}")
print(f"type    : {dim_type}")
print(f"encoding: {encoding}")

if encoding == "pentadal":
    print("Encoding confirmed as 'pentadal'.")
else:
    # Graceful fallback: accept any temporal dimension and document actual encoding
    KNOWN_TEMPORAL = {"dekadal", "pentadal", "monthly"}
    is_temporal = encoding in KNOWN_TEMPORAL or "temporal" in dim_type.lower()
    assert is_temporal, (
        f"Expected a temporal dimension; got encoding={encoding!r} type={dim_type!r}"
    )
    print(f"Note: encoding is {encoding!r}, not 'pentadal'. Adjust COLLECTION_ID if needed.")

## Step 2 — Page through all 72 periods

Fetch pentadal children for year 2024 in pages of 12, following `links[rel=next]`
until no next link is present. The loop accumulates all periods and asserts the total
equals 72 (6 pentads × 12 months).

The `offset`-based cursor mirrors the OGC API — Features paging contract: the server
embeds the next page URL in `links`. Clients must follow the link rather than
computing offsets independently, in case the server uses cursor-based paging.

In [ ]:
all_periods = []
url = f"{DIM_BASE}/time/children"
params = {"year": 2024, "limit": 12, "offset": 0}
page_num = 0

while True:
    resp = client.get(url, params=params)
    assert resp.status_code == 200, (
        f"Page {page_num}: Expected 200, got {resp.status_code}: {resp.text}"
    )
    data = resp.json()
    page_members = data.get("members", [])
    all_periods.extend(page_members)
    page_num += 1
    print(f"  Page {page_num}: {len(page_members)} periods fetched (running total: {len(all_periods)})")

    next_link = next(
        (lnk for lnk in data.get("links", []) if lnk.get("rel") == "next"),
        None,
    )
    if not next_link:
        break

    # Follow the next link; reset params if the href is absolute
    next_href = next_link.get("href", "")
    if "?" in next_href:
        # Server embedded full URL with query string — use it directly
        from urllib.parse import urlparse, parse_qs
        parsed = urlparse(next_href)
        qs = parse_qs(parsed.query)
        params = {k: v[0] for k, v in qs.items()}
        url = parsed.path
    else:
        params["offset"] += params.get("limit", 12)

assert len(all_periods) == 72, f"Expected 72 pentadal periods for 2024, got {len(all_periods)}"
print(f"\nAll {len(all_periods)} pentadal periods retrieved across {page_num} page(s).")

## Step 3 — Verify February edge case

In a pentadal calendar, February has 6 pentads (P1–P6) like every other month, but P6
is variable: it covers day 26 to the end of the month (28 days common year / 29 days
leap year). 2024 is a leap year, so February P6 spans Feb 26–29 (4 days instead of 5).

The API must return exactly 6 periods for February regardless of year type.

In [ ]:
# Filter the periods already collected for February (month 02)
feb_periods = [
    p for p in all_periods
    if str(p.get("id", "")).startswith("2024-02") or (
        isinstance(p.get("interval"), list)
        and p["interval"]
        and str(p["interval"][0]).startswith("2024-02")
    )
]

print(f"February 2024 pentads ({len(feb_periods)}):")
for p in feb_periods:
    print(f"  id={p.get('id')}  interval={p.get('interval')}")

assert len(feb_periods) == 6, (
    f"Expected 6 pentadal periods for February 2024, got {len(feb_periods)}"
)

# Locate P6 and verify its end date for leap-year correctness
p6 = next(
    (p for p in feb_periods if str(p.get("id", "")).endswith("P6")),
    None,
)
if p6 and p6.get("interval"):
    end_date = p6["interval"][1] or ""
    print(f"\nFebruary P6 end date: {end_date}")
    # 2024 leap year: P6 must end on Feb 29
    assert "02-29" in end_date or "02-28" in end_date, (
        f"February P6 end date unexpected for 2024: {end_date}"
    )
    print("Leap-year February P6 end date verified.")
else:
    print("P6 interval not present — skipping end-date assertion.")

## Edge cases

### Oversize limit — server must clamp, not reject

Passing a `limit` larger than the server maximum (typically 100 or 200) must not
produce a 422 Unprocessable Entity. The server must silently clamp the value and
return a valid page. This prevents brittle client code that hard-codes server limits.

In [ ]:
resp = client.get(
    f"{DIM_BASE}/time/children",
    params={"year": 2024, "limit": 99999},
)
print(resp.status_code)
assert resp.status_code != 422, (
    f"Server must clamp oversize limit, not reject with 422; got {resp.status_code}: {resp.text}"
)
assert resp.status_code == 200, (
    f"Expected 200 for oversize limit (clamped), got {resp.status_code}: {resp.text}"
)

data = resp.json()
clamped_members = data.get("members", [])
print(f"Members returned with limit=99999: {len(clamped_members)}")

# The server must return at most 72 periods (full year) — never more
assert len(clamped_members) <= 72, (
    f"Server returned {len(clamped_members)} > 72 periods — clamping appears to have failed"
)
print(f"Clamping confirmed: {len(clamped_members)} periods returned (≤ 72).")

client.close()